# Assignment 3 — ACID Properties Demo
## Campus Trading Application · B+ Tree Engine

This notebook demonstrates all four ACID properties on the **campus trading**
database.  Every table is backed exclusively by a B+ Tree via `DatabaseManager`.

### Four-table schema

| Table | Primary key | Role |
|-------|-------------|------|
| **Listing** | `ListingID` | Item a seller has posted for sale |
| **Offer** | `OfferID` | A buyer's bid on a Listing |
| **Transaction** | `TransactionID` | Finalised deal (Accepted offer) |
| **Notification** | `NotificationID` | System messages to buyers / sellers |

### Atomic workflow: `accept_offer_atomic`
One call touches **all four tables** in a single WAL-backed transaction:
1. Mark chosen Offer **Accepted** (update Offer)
2. Decline competing offers on the same Listing (update Offer)
3. Mark Listing **Sold** (update Listing)
4. Insert **Transaction** row
5. Insert **Notification** rows for buyer and seller

---

In [1]:
from __future__ import annotations

import os
import sys
import threading
import time
from copy import deepcopy
from pathlib import Path

# Locate Module_A so imports work regardless of notebook CWD
cwd = Path.cwd()
module_a_root = cwd
if not (module_a_root / "database").exists():
    for p in [cwd, *cwd.parents]:
        if (p / "database").exists():
            module_a_root = p
            break
if str(module_a_root) not in sys.path:
    sys.path.insert(0, str(module_a_root))

from database import DatabaseManager, RecoveryManager
from database.campus_schema import (
    SeedProfile,
    install_campus_schema,
    seed_campus_tables_transactional,
    CAMPUS_TABLE_NAMES,
)
from database.campus_workflow import accept_offer_atomic
from database.table_display import snapshot_for_ipython

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str

# Artifacts folder
demo_dir = module_a_root / "artifacts" / "acid_demo"
demo_dir.mkdir(parents=True, exist_ok=True)

print(f"Module A root : {module_a_root}")
print(f"Demo artifacts: {demo_dir}")
print(f"Campus tables : {CAMPUS_TABLE_NAMES}")

Module A root : /Users/bhavikpatel/Downloads/CourseWork/Database/Campus_Trading_App/Assignment_3/Module_A
Demo artifacts: /Users/bhavikpatel/Downloads/CourseWork/Database/Campus_Trading_App/Assignment_3/Module_A/artifacts/acid_demo
Campus tables : ('Offer', 'Listing', 'Transaction', 'Notification')


## 1. Install schema and seed data

We use a `SeedProfile` with **1 listing** and **2 buyers** so the ACID
scenarios are clear and easy to follow.  All seed rows are inserted through
`seed_campus_tables_transactional` — every insert goes through `tx_insert`
and is logged to the WAL so recovery can replay the full initial state.

In [2]:
PROFILE = SeedProfile(
    db_name="campus",
    seller_id=10,
    buyer_ids=[101, 102],   # two buyers competing on the same listing
    listing_base_id=1000,
    offer_base_id=2000,
    listings_per_run=1,     # one listing keeps the output concise
)
DB = PROFILE.db_name

MAIN_WAL = str(demo_dir / "campus_acid_wal.log")
if os.path.exists(MAIN_WAL):
    os.remove(MAIN_WAL)

dbm = DatabaseManager(wal_path=MAIN_WAL)
install_campus_schema(dbm, PROFILE)
seed_campus_tables_transactional(dbm, PROFILE)

display(Markdown("### Initial state — four campus tables (after WAL-logged seed)"))
display(snapshot_for_ipython(dbm, DB))

### Initial state — four campus tables (after WAL-logged seed)


## Snapshot: `campus`


### Offer
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| OfferID | ListingID | BuyerID | OfferedPrice | AgreedPrice | OfferStatus | Reason | ResponseDate |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| 2000    | 1000      | 101     | 90.0         | 0.0         | Submitted   |        |              |
| 2001    | 1000      | 102     | 95.0         | 0.0         | Submitted   |        |              |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+


### Listing
+-----------+----------+--------+-------------+------------------+
| ListingID | SellerID | Status | AskingPrice | LastModifiedDate |
+-----------+----------+--------+-------------+------------------+
| 1000      | 10       | Listed | 100.0       |                  |
+-----------+----------+--------+-------------+------------------+


### Transaction
*(no rows)*


### Notification
*(no rows)*


### Utility helpers

`deep_snapshot` captures the full in-memory state of all four tables as a
Python dict of sorted lists.  Comparing two snapshots with `==` is how we
prove atomicity (rollback restores exact prior state) and durability (WAL
replay restores exact committed state).

In [4]:
def deep_snapshot(db: DatabaseManager, profile: SeedProfile = PROFILE) -> dict:
    """Return sorted (key, record) pairs for every campus table."""
    out = {}
    for name in CAMPUS_TABLE_NAMES:
        t, _ = db.get_table(profile.db_name, name)
        out[name] = sorted([(k, deepcopy(v)) for k, v in t.get_all()], key=lambda x: x[0])
    return out


def show(db: DatabaseManager, title: str) -> None:
    display(Markdown(f"### {title}"))
    display(snapshot_for_ipython(db, DB))

---

## 2. Atomicity

> **Requirement (spec):** Crash during a multi-table transaction and verify rollback.

### 2a. Successful commit (all-or-nothing)

`accept_offer_atomic` commits when everything succeeds: Offer updated,
Listing marked Sold, Transaction and Notification rows inserted — **all four
tables change together**.

In [ ]:
snap_before = deep_snapshot(dbm)

ok, msg = accept_offer_atomic(
    dbm, DB,
    offer_id=2000,           # buyer 101's offer on listing 1000
    acting_seller_id=10,
    include_notifications=True,
    create_declined_transactions=True,
)
print(f"accept_offer_atomic → ok={ok}, msg='{msg}'")
assert ok, "Expected success"

show(dbm, "Atomicity — SUCCESS: all four tables updated in one commit")

# Verify each expected change
offer_t, _ = dbm.get_table(DB, "Offer")
listing_t, _ = dbm.get_table(DB, "Listing")
txn_t, _ = dbm.get_table(DB, "Transaction")
notif_t, _ = dbm.get_table(DB, "Notification")

assert offer_t.get(2000)["OfferStatus"] == "Accepted",  "Offer 2000 must be Accepted"
assert offer_t.get(2001)["OfferStatus"] == "Declined",  "Offer 2001 must be Declined"
assert listing_t.get(1000)["Status"] == "Sold",          "Listing 1000 must be Sold"
assert len(txn_t.get_all()) >= 1,                        "At least one Transaction row"
assert len(notif_t.get_all()) >= 2,                      "At least two Notification rows"
print("✓ All four-table checks pass for successful commit.")

### 2b. Mid-transaction failure → full rollback

We reset the database and try again with `fail_after_step=3`.
This injects a crash **after Offer and Listing have been updated** but
**before the Transaction/Notification rows are inserted**.  The engine must
undo all partial changes so every table returns to its exact pre-call state.

In [5]:
# Fresh database for rollback demo (independent WAL)
ATOM_WAL = str(demo_dir / "atomicity_wal.log")
if os.path.exists(ATOM_WAL):
    os.remove(ATOM_WAL)

dbm_a = DatabaseManager(wal_path=ATOM_WAL)
install_campus_schema(dbm_a, PROFILE)
seed_campus_tables_transactional(dbm_a, PROFILE)

snap_pre_fail = deep_snapshot(dbm_a)
show(dbm_a, "Atomicity — BEFORE injected failure")

ok, msg = accept_offer_atomic(
    dbm_a, DB,
    offer_id=2000,
    acting_seller_id=10,
    include_notifications=True,
    create_declined_transactions=True,
    fail_after_step=3,      # crash after Listing marked Sold, before Transaction insert
)
print(f"accept_offer_atomic (fail_after_step=3) → ok={ok}, msg='{msg}'")
assert not ok, "Expected failure"

snap_post_fail = deep_snapshot(dbm_a)
show(dbm_a, "Atomicity — AFTER rollback (must match pre-failure snapshot)")

assert snap_post_fail == snap_pre_fail, "ROLLBACK must restore exact pre-transaction state"
print("✓ deep_snapshot after rollback == snapshot before failed transaction.")
print("  Atomicity: no partial updates remain in any of the four tables.")

### Atomicity — BEFORE injected failure


## Snapshot: `campus`


### Offer
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| OfferID | ListingID | BuyerID | OfferedPrice | AgreedPrice | OfferStatus | Reason | ResponseDate |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| 2000    | 1000      | 101     | 90.0         | 0.0         | Submitted   |        |              |
| 2001    | 1000      | 102     | 95.0         | 0.0         | Submitted   |        |              |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+


### Listing
+-----------+----------+--------+-------------+------------------+
| ListingID | SellerID | Status | AskingPrice | LastModifiedDate |
+-----------+----------+--------+-------------+------------------+
| 1000      | 10       | Listed | 100.0       |                  |
+-----------+----------+--------+-------------+------------------+


### Transaction
*(no rows)*


### Notification
*(no rows)*


accept_offer_atomic (fail_after_step=3) → ok=False, msg='Injected failure after step 3'


### Atomicity — AFTER rollback (must match pre-failure snapshot)


## Snapshot: `campus`


### Offer
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| OfferID | ListingID | BuyerID | OfferedPrice | AgreedPrice | OfferStatus | Reason | ResponseDate |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+
| 2000    | 1000      | 101     | 90.0         | 0.0         | Submitted   |        |              |
| 2001    | 1000      | 102     | 95.0         | 0.0         | Submitted   |        |              |
+---------+-----------+---------+--------------+-------------+-------------+--------+--------------+


### Listing
+-----------+----------+--------+-------------+------------------+
| ListingID | SellerID | Status | AskingPrice | LastModifiedDate |
+-----------+----------+--------+-------------+------------------+
| 1000      | 10       | Listed | 100.0       |                  |
+-----------+----------+--------+-------------+------------------+


### Transaction
*(no rows)*


### Notification
*(no rows)*


✓ deep_snapshot after rollback == snapshot before failed transaction.
  Atomicity: no partial updates remain in any of the four tables.


---

## 3. Consistency

> **Requirement (spec):** All relations remain valid after operations.

Consistency means the engine rejects invalid state transitions **without
leaving any partial corruption**.  We test two invalid cases:

1. Wrong seller ID — the caller is not the listing owner.
2. Offer already `Accepted` — it is no longer `Submitted`.

Both must return `(False, msg)` and leave all four tables **unchanged**.

In [ ]:
CONS_WAL = str(demo_dir / "consistency_wal.log")
if os.path.exists(CONS_WAL):
    os.remove(CONS_WAL)

dbm_c = DatabaseManager(wal_path=CONS_WAL)
install_campus_schema(dbm_c, PROFILE)
seed_campus_tables_transactional(dbm_c, PROFILE)
snap_cons = deep_snapshot(dbm_c)

# Case 1: wrong seller
ok1, msg1 = accept_offer_atomic(dbm_c, DB, offer_id=2000, acting_seller_id=9999)
print(f"Case 1 — wrong seller  : ok={ok1}, msg='{msg1}'")
assert not ok1 and "listing owner" in msg1
assert deep_snapshot(dbm_c) == snap_cons, "Snapshot must be unchanged"

# Case 2: mark offer non-Submitted first, then try to accept it
offer_t, _ = dbm_c.get_table(DB, "Offer")
row = dict(offer_t.get(2000))
row["OfferStatus"] = "Declined"
offer_t.update(2000, row)   # direct update (not WAL-logged; fine for consistency demo)

ok2, msg2 = accept_offer_atomic(dbm_c, DB, offer_id=2000, acting_seller_id=10)
print(f"Case 2 — non-Submitted : ok={ok2}, msg='{msg2}'")
assert not ok2 and "no longer active" in msg2

# Restore offer for a clean display
row["OfferStatus"] = "Submitted"
offer_t.update(2000, row)

show(dbm_c, "Consistency — tables unchanged after both invalid attempts")
print("✓ Both invalid accepts rejected; no silent corruption in any table.")

---

## 4. Isolation

> **Requirement (spec):** Concurrent transactions on the same data — no visible intermediate states, no corruption.

The engine uses a **global serial lock** (`_serial_lock`) inside every
transactional path, providing **serializable isolation**: concurrent threads
execute strictly one at a time with no interleaving.

**Scenario:** Two threads simultaneously try to accept **different offers**
on the **same listing** (a classic race condition).

- Exactly **one** thread commits; the listing is **Sold**.
- The other thread loses the race and is rejected (listing no longer
  available after the winner commits).
- We record monotonic timestamps to show the execution windows of the two
  transactions **cannot overlap** — confirming serialized execution.

In [ ]:
ISO_WAL = str(demo_dir / "isolation_wal.log")
if os.path.exists(ISO_WAL):
    os.remove(ISO_WAL)

dbm_i = DatabaseManager(wal_path=ISO_WAL)
install_campus_schema(dbm_i, PROFILE)
seed_campus_tables_transactional(dbm_i, PROFILE)

results: list[tuple[str, bool, str]] = []
timeline: list[tuple[str, float]] = []
lock_r = threading.Lock()
lock_t = threading.Lock()
barrier = threading.Barrier(2)


def race_runner(label: str, offer_id: int) -> None:
    barrier.wait()          # both threads start simultaneously
    with lock_t:
        timeline.append((f"{label}_enter", time.monotonic()))
    ok, msg = accept_offer_atomic(
        dbm_i, DB,
        offer_id=offer_id,
        acting_seller_id=PROFILE.seller_id,
        include_notifications=True,
        create_declined_transactions=True,
    )
    with lock_r:
        results.append((label, ok, msg))
    with lock_t:
        timeline.append((f"{label}_exit", time.monotonic()))


t1 = threading.Thread(target=race_runner, args=("Thread-A", 2000))  # buyer 101
t2 = threading.Thread(target=race_runner, args=("Thread-B", 2001))  # buyer 102
t1.start()
t2.start()
t1.join()
t2.join()

print("Isolation — concurrent race results:")
for label, ok, msg in results:
    print(f"  {label}: ok={ok}, msg='{msg}'")

successes = [r for r in results if r[1]]
failures  = [r for r in results if not r[1]]
assert len(successes) == 1, f"Exactly one winner expected, got {len(successes)}"
assert len(failures)  == 1, f"Exactly one loser expected,  got {len(failures)}"

# Verify referential integrity
offer_t, _ = dbm_i.get_table(DB, "Offer")
listing_t, _ = dbm_i.get_table(DB, "Listing")
accepted = [r for _, r in offer_t.get_all() if r["OfferStatus"] == "Accepted"]
assert len(accepted) == 1, "Exactly one Accepted offer"
assert listing_t.get(PROFILE.listing_base_id)["Status"] == "Sold"

# Timeline: prove the critical sections did not interleave
ev = {k: v for k, v in timeline}
a_enter, a_exit = ev["Thread-A_enter"], ev["Thread-A_exit"]
b_enter, b_exit = ev["Thread-B_enter"], ev["Thread-B_exit"]
non_overlapping = (a_exit <= b_enter) or (b_exit <= a_enter)
print(f"\nTimeline (monotonic seconds):")
print(f"  Thread-A: enter={a_enter:.6f}  exit={a_exit:.6f}")
print(f"  Thread-B: enter={b_enter:.6f}  exit={b_exit:.6f}")
print(f"  Serialized (non-overlapping lock windows): {non_overlapping}")

show(dbm_i, "Isolation — final state after concurrent race")
print("✓ Exactly one winner, listing Sold, no corruption — isolation confirmed.")

---

## 5. Durability

> **Requirement (spec):** Restart system and verify committed data persists.

Durability is demonstrated by **simulating a process crash** (discarding
all in-memory B+ Trees) and then replaying the WAL file into a brand-new
`DatabaseManager`.  The recovered state must be **bit-for-bit identical**
to the committed state before the crash.

### Steps
1. Seed (WAL-logged) + accept one offer → commit → **snapshot A**.
2. **Crash**: create a new `DatabaseManager` with the same WAL file; all
   B+ Trees are empty (simulates memory loss).
3. **Recover**: reinstall schema, call `RecoveryManager.recover_into` to
   replay REDO entries from the WAL.
4. **snapshot B** from recovered manager → assert `A == B`.

In [ ]:
DUR_WAL = str(demo_dir / "durability_wal.log")
if os.path.exists(DUR_WAL):
    os.remove(DUR_WAL)

# ── Step 1: live system — seed + commit an accept ─────────────────────────
db_live = DatabaseManager(wal_path=DUR_WAL)
install_campus_schema(db_live, PROFILE)
seed_campus_tables_transactional(db_live, PROFILE)

ok, msg = accept_offer_atomic(
    db_live, DB,
    offer_id=2000,
    acting_seller_id=PROFILE.seller_id,
    include_notifications=True,
    create_declined_transactions=True,
)
assert ok, f"Setup failed: {msg}"

snapshot_A = deep_snapshot(db_live, PROFILE)
show(db_live, "Durability — BEFORE crash (committed state, snapshot A)")

# ── Step 2: simulated crash ───────────────────────────────────────────────
print("--- Simulating crash: discarding all in-memory B+ Trees ---")
del db_live   # memory gone; WAL file on disk survives

# ── Step 3: recovery on restart ───────────────────────────────────────────
db_recovered = DatabaseManager(wal_path=DUR_WAL)   # fresh manager, empty tables
install_campus_schema(db_recovered, PROFILE)         # reinstall empty schema

rec = RecoveryManager(db_recovered.wal)
recovery_summary = rec.recover_into(db_recovered)
print("Recovery summary:")
print(f"  REDO transactions : {recovery_summary['redo_transactions']}")
print(f"  UNDO transactions : {recovery_summary['undo_transactions']}")
print(f"  Applied REDO ops  : {recovery_summary['applied_redo']}")
print(f"  Applied UNDO ops  : {recovery_summary['applied_undo']}")

# ── Step 4: compare snapshots ─────────────────────────────────────────────
snapshot_B = deep_snapshot(db_recovered, PROFILE)
show(db_recovered, "Durability — AFTER recovery (snapshot B must equal A)")

assert snapshot_B == snapshot_A, (
    "Durability FAILED: recovered snapshot does not match committed snapshot!"
)
print("✓ Snapshot A (pre-crash) == Snapshot B (post-recovery).")
print("  Durability: committed data survived the simulated crash and restart.")

---

## Summary

| Property | Demonstrated by | Result |
|----------|----------------|--------|
| **Atomicity** | `accept_offer_atomic` with `fail_after_step=3`; `deep_snapshot` before == after rollback | ✓ |
| **Consistency** | Wrong seller / non-Submitted offer both rejected; tables unchanged | ✓ |
| **Isolation** | Two threads race on same listing; exactly one winner; non-overlapping lock windows | ✓ |
| **Durability** | WAL replay on fresh `DatabaseManager`; `deep_snapshot` A == B | ✓ |

All four properties hold on the **campus trading** schema (Offer, Listing,
Transaction, Notification) backed by B+ Trees with WAL-based recovery.